# Research focus rule ? FS03 fixed params

Notebook n?y reproduce lu?ng c?a `research/rf_mlflow/reproduce_fs03_fixed_params.py` theo phong c?ch notebook: m?i cell l?m m?t vi?c r? r?ng v? c? output ?? ki?m tra nhanh.

L?u ?: notebook kh?ng import file `reproduce_fs03_fixed_params.py`. C?c h?ng s? v? helper c?n thi?t ???c vi?t tr?c ti?p trong notebook.


## 1. Import th? vi?n

In [4]:
from __future__ import annotations

import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

## 2. Tr? path project

Cell n?y gi?p notebook ch?y ???c d? m? t? th? m?c `notebooks` hay t? project root.

In [5]:
NOTEBOOK_PATH = Path(r"D:\ML_Trade_Bot\notebooks\research_focus_rule.ipynb")
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "research").exists():
    PROJECT_ROOT = Path(r"D:\ML_Trade_Bot")

RESEARCH_DIR = PROJECT_ROOT / "research"
RF_MLFLOW_DIR = RESEARCH_DIR / "rf_mlflow"

for path in [RESEARCH_DIR, RF_MLFLOW_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESEARCH_DIR:", RESEARCH_DIR)
print("RF_MLFLOW_DIR:", RF_MLFLOW_DIR)

PROJECT_ROOT: D:\ML_Trade_Bot
RESEARCH_DIR: D:\ML_Trade_Bot\research
RF_MLFLOW_DIR: D:\ML_Trade_Bot\research\rf_mlflow


## 3. Import module chung c?a project

Kh?ng import `reproduce_fs03_fixed_params.py`.

In [6]:
from research.config import CFG, PROJECT_ROOT as CONFIG_PROJECT_ROOT, ensure_dirs
from research.features import RobustClipScaler, named_feature_sets
from research.mophong_adapter import orders_from_scores, simulate_orders
from research.plots import plot_model_diagnostics, plot_threshold_curve, plot_yearly_mophong
from research.rf_mlflow.train_rf_mlflow import _candidate_dataset, _proxy_threshold_table, _score_full_frame, build_cached_dataset

PROJECT_ROOT = CONFIG_PROJECT_ROOT
ensure_dirs()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("outputs_dir :", CFG.outputs_dir)
print("model_dir   :", CFG.model_dir)
print("experiment  :", CFG.experiment_name)

LogPath: D:\ML_Trade_Bot\notebooksSCJ999/log/2026-09-14/method.MoPhongDeals_info.log
LogPath: D:\ML_Trade_Bot\notebooksSCJ999/log/2026-09-14/MyLogger_info.log
PROJECT_ROOT: D:\ML_Trade_Bot
outputs_dir : D:\ML_Trade_Bot\research\rf_mlflow\outputs
model_dir   : D:\ML_Trade_Bot\research\rf_mlflow\models
experiment  : RF_MLFlow_XAUUSD_M1_Buy_WinLose


## 4. C?u h?nh experiment

Ch?nh c?c gi? tr? trong cell n?y tr??c khi ch?y to?n notebook.

In [7]:
FEATURE_SET = "fs03_lags_cycle"
SOURCE_RUN_NAME = "research10_1788533519"
SOURCE_RUN_DIR = CFG.outputs_dir / SOURCE_RUN_NAME

SIMULATE_THRESHOLDS = [0.56, 0.57, 0.58]
FORCE_DATASET = False
SKIP_SIMULATE = False
RANDOM_STATE = 1003

FIXED_PARAMS = {
    "n_estimators": 220,
    "min_samples_split": 1200,
    "min_samples_leaf": 250,
    "max_samples": 0.85,
    "max_features": 0.70,
    "max_depth": None,
    "class_weight": None,
}

pd.Series({
    "feature_set": FEATURE_SET,
    "source_run_dir": str(SOURCE_RUN_DIR),
    "simulate_thresholds": SIMULATE_THRESHOLDS,
    "force_dataset": FORCE_DATASET,
    "skip_simulate": SKIP_SIMULATE,
    "random_state": RANDOM_STATE,
})

feature_set                                              fs03_lags_cycle
source_run_dir         D:\ML_Trade_Bot\research\rf_mlflow\outputs\res...
simulate_thresholds                                   [0.56, 0.57, 0.58]
force_dataset                                                      False
skip_simulate                                                      False
random_state                                                        1003
dtype: object

## 5. T?o th? m?c output cho run hi?n t?i

In [8]:
started = time.time()
run_id = int(started)

out_dir = CFG.outputs_dir / f"reproduce_fs03_fixed_params_{run_id}"
chart_dir = out_dir / "charts"
out_dir.mkdir(parents=True, exist_ok=True)
chart_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "valid_curve": out_dir / "fs03_fixed_threshold_valid.csv",
    "test_curve": out_dir / "fs03_fixed_threshold_test.csv",
    "comparison": out_dir / "fs03_fixed_vs_research10_1788533519_test_curve.csv",
    "yearly": out_dir / "fs03_fixed_mophong_thresholds_yearly.csv",
    "sim_total": out_dir / "fs03_fixed_mophong_thresholds_total.csv",
    "summary": out_dir / "fs03_fixed_summary.json",
    "model": CFG.model_dir / f"fs03_fixed_params_{run_id}_bundle.joblib",
}

print("run_id:", run_id)
print("out_dir:", out_dir)
print("chart_dir:", chart_dir)

run_id: 1789400402
out_dir: D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402
chart_dir: D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts


## 6. Helper metric

In [9]:
def metrics_table(y, p) -> dict:
    pred = (p >= 0.5).astype(int)
    out = {
        "samples": int(len(y)),
        "positive_rate": float(np.mean(y)) if len(y) else 0.0,
        "accuracy": float(accuracy_score(y, pred)) if len(y) else 0.0,
    }
    if len(np.unique(y)) == 2:
        out["auc"] = float(roc_auc_score(y, p))
        out["brier"] = float(brier_score_loss(y, p))
        out["logloss"] = float(log_loss(y, p, labels=[0, 1]))
    return out

## 7. Helper threshold curve

In [27]:
def total_curve(curve: pd.DataFrame) -> pd.DataFrame:
    out = curve.groupby("threshold", as_index=False).agg(
        resolved=("resolved", "sum"),
        wins=("wins", "sum"),
        losses=("losses", "sum"),
    )
    out["winrate"] = (out["wins"] * 100 / out["resolved"].replace(0, np.nan)).round(4)
    return out


def compare_with_source_run(new_curve: pd.DataFrame) -> pd.DataFrame:
    old_curve_path = SOURCE_RUN_DIR / "fs03_lags_cycle_threshold_test.csv"
    old_curve = pd.read_csv(old_curve_path)

    old_total = total_curve(old_curve).rename(
        columns={
            "resolved": "old_resolved",
            "wins": "old_wins",
            "losses": "old_losses",
            "winrate": "old_winrate",
        }
    )
    new_total = total_curve(new_curve).rename(
        columns={
            "resolved": "new_resolved",
            "wins": "new_wins",
            "losses": "new_losses",
            "winrate": "new_winrate",
        }
    )

    cmp = old_total.merge(new_total, on="threshold", how="outer").sort_values("threshold")
    cmp["resolved_diff"] = cmp["new_resolved"] - cmp["old_resolved"]
    cmp["winrate_diff"] = cmp["new_winrate"] - cmp["old_winrate"]
    return cmp

## 8. Load dataset cache

In [12]:
frame = build_cached_dataset(force=FORCE_DATASET).sort_values("dates").reset_index(drop=True)

print("shape:", frame.shape)
print("date range:", frame["dates"].min(), "?", frame["dates"].max())
display(frame.head())

Loading year: 2018
Loading year: 2019
Loading year: 2020
Loading year: 2021
Loading year: 2022
Loading year: 2023
Loading year: 2024
Loading year: 2025
Loading year: 2026
shape: (2988871, 492)
date range: 2018-01-02 06:00:00 ? 2026-09-11 20:57:00


,dates,open,low,high,close,ret_1,hl_range,upper_wick,lower_wick,body_abs,...,sig_pullback_trend_buy,sig_momentum_buy,buy_signal,label,label_res,entry_index,entry_price,year,tick_volume,volume_z_100
0,2018-01-02 06:00:00,1307.421,1307.400,1307.422,1307.400,-2.1,2.2,0.1,0.0,2.1,...,0,0,0,1.0,Win,1,1307.397,2018,NaN,NaN
1,2018-01-02 06:01:00,1307.397,1307.393,1307.608,1307.603,20.6,21.5,0.5,0.4,20.6,...,0,0,0,1.0,Win,2,1307.597,2018,NaN,NaN
2,2018-01-02 06:02:00,1307.597,1307.590,1307.597,1307.590,-0.7,0.7,0.0,0.0,0.7,...,0,0,0,1.0,Win,3,1307.590,2018,NaN,NaN
3,2018-01-02 06:03:00,1307.590,1307.584,1308.160,1307.690,10.0,57.6,47.0,0.6,10.0,...,0,0,0,1.0,Win,4,1307.690,2018,NaN,NaN
4,2018-01-02 06:04:00,1307.690,1307.690,1308.022,1308.021,33.1,33.2,0.1,0.0,33.1,...,0,0,0,1.0,Win,5,1308.021,2018,NaN,NaN


## 9. Ch?n feature set

In [13]:
feature_sets = named_feature_sets(frame)
cols = feature_sets[FEATURE_SET]

print("feature_set:", FEATURE_SET)
print("n_features:", len(cols))
display(pd.DataFrame({"feature": cols}).head(30))

feature_set: fs03_lags_cycle
n_features: 16


,feature
0,close_diff_lag_5
1,body_lag_5
2,close_diff_lag_8
3,body_lag_8
4,close_diff_lag_13
5,body_lag_13
6,close_diff_lag_21
7,body_lag_21
8,tod_sin
9,tod_cos


## 10. T?ch train / valid / test

In [20]:
x_train_raw, y_train, train_frame = _candidate_dataset(frame, cols, CFG.split.train_years)
x_valid_raw, y_valid, valid_frame = _candidate_dataset(frame, cols, CFG.split.valid_years)
x_test_raw, y_test, test_frame = _candidate_dataset(frame, cols, CFG.split.test_years)

split_info = pd.DataFrame([
    {"split": "train", "years": list(CFG.split.train_years), "rows": len(y_train), "positive_rate": float(np.mean(y_train))},
    {"split": "valid", "years": list(CFG.split.valid_years), "rows": len(y_valid), "positive_rate": float(np.mean(y_valid))},
    {"split": "test", "years": list(CFG.split.test_years), "rows": len(y_test), "positive_rate": float(np.mean(y_test))},
])
display(split_info)

,split,years,rows,positive_rate
0,train,"[2018, 2019, 2020, 2021, 2022]",678345,0.502401
1,valid,[2023],145013,0.498355
2,test,"[2024, 2025, 2026]",369504,0.517705


## 11. Fit scaler

In [21]:
scaler = RobustClipScaler().fit(x_train_raw)

x_train = scaler.transform(x_train_raw)
x_valid = scaler.transform(x_valid_raw)
x_test = scaler.transform(x_test_raw)

print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("x_test :", x_test.shape)

x_train: (678345, 16)
x_valid: (145013, 16)
x_test : (369504, 16)


## 12. Train RandomForest v?i fixed params

In [23]:
model = RandomForestClassifier(
    **FIXED_PARAMS,
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

model.fit(x_train, y_train)
model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",220
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",1200
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",250
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.7
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1003
,"max_samples max_samples: int or float, default=NoneIf bootstrap is True, the number of samples to draw from Xto train each base estimator.- If None (default), then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples... versionadded:: 0.22.. versionchanged:: 1.9 Float `max_samples` is relative to `sample_weight.sum()` instead of `X.shape[0]` for weighted samples.",0.85
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0


## 13. Evaluate valid / test

In [24]:
p_valid = model.predict_proba(x_valid)[:, 1]
p_test = model.predict_proba(x_test)[:, 1]

valid_metrics = metrics_table(y_valid, p_valid)
test_metrics = metrics_table(y_test, p_test)

metrics_df = pd.DataFrame([valid_metrics, test_metrics], index=["valid", "test"])
display(metrics_df)

,samples,positive_rate,accuracy,auc,brier,logloss
valid,145013,0.498355,0.528842,0.541901,0.248595,0.690330
test,369504,0.517705,0.506203,0.511379,0.250209,0.693567


## 14. Score to?n b? frame

In [25]:
scores = _score_full_frame(frame, cols, scaler, model)

score_summary = pd.Series(scores).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(score_summary)

count    2.988673e+06
mean     5.026963e-01
std      3.311112e-02
min      3.604829e-01
1%       4.254509e-01
5%       4.503354e-01
50%      5.016850e-01
95%      5.592627e-01
99%      5.790718e-01
max      6.429191e-01
dtype: float64

## 15. Build threshold curves

In [28]:
valid_curve = _proxy_threshold_table(frame, scores, CFG.split.valid_years)
test_curve = _proxy_threshold_table(frame, scores, CFG.split.test_years)
#comparison = compare_with_source_run(test_curve)

valid_curve.to_csv(paths["valid_curve"], index=False, encoding="utf-8-sig")
test_curve.to_csv(paths["test_curve"], index=False, encoding="utf-8-sig")
#comparison.to_csv(paths["comparison"], index=False, encoding="utf-8-sig")

print("saved:", paths["valid_curve"])
print("saved:", paths["test_curve"])
#print("saved:", paths["comparison"])

display(total_curve(valid_curve).head(10))
display(total_curve(test_curve).head(10))

saved: D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\fs03_fixed_threshold_valid.csv
saved: D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\fs03_fixed_threshold_test.csv


,threshold,resolved,wins,losses,winrate
0,0.50,55618,29300,26318,52.6808
1,0.51,40711,22035,18676,54.1254
2,0.52,29299,16254,13045,55.4763
3,0.53,21068,11959,9109,56.7638
4,0.54,14831,8636,6195,58.2294
5,0.55,9877,5865,4012,59.3804
6,0.56,5964,3583,2381,60.0771
7,0.57,3131,1880,1251,60.0447
8,0.58,1195,713,482,59.6653
9,0.59,405,228,177,56.2963


,threshold,resolved,wins,losses,winrate
0,0.50,137125,72659,64466,52.9874
1,0.51,101230,54252,46978,53.5928
2,0.52,72526,39319,33207,54.2137
3,0.53,51067,28034,23033,54.8965
4,0.54,34007,18740,15267,55.1063
5,0.55,20698,11599,9099,56.0392
6,0.56,11262,6387,4875,56.7128
7,0.57,5287,3077,2210,58.1994
8,0.58,1841,1101,740,59.8045
9,0.59,563,352,211,62.5222


## 16. So s?nh threshold quan tr?ng v?i source run

In [29]:
key_threshold_comparison = comparison[comparison["threshold"].isin(SIMULATE_THRESHOLDS)].copy()
display(key_threshold_comparison)

NameError: name 'key_threshold_comparison' is not defined

## 17. L?u model bundle

In [30]:
model_bundle = {
    "model": model,
    "scaler": scaler,
    "feature_columns": cols,
    "params": FIXED_PARAMS,
}

joblib.dump(model_bundle, paths["model"], compress=3)
print("saved:", paths["model"])

saved: D:\ML_Trade_Bot\research\rf_mlflow\models\fs03_fixed_params_1789400402_bundle.joblib


## 18. V? diagnostic charts

In [31]:
chart_paths = []
chart_paths += plot_model_diagnostics(y_valid, p_valid, chart_dir, "fs03_fixed_valid")
chart_paths.append(
    plot_threshold_curve(valid_curve, chart_dir / "fs03_fixed_threshold_valid.png", "fs03 fixed validation threshold")
)
chart_paths.append(
    plot_threshold_curve(test_curve, chart_dir / "fs03_fixed_threshold_test.png", "fs03 fixed test threshold")
)

for path in chart_paths:
    print(path)

D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts\fs03_fixed_valid_roc.png
D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts\fs03_fixed_valid_confusion.png
D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts\fs03_fixed_valid_calibration.png
D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts\fs03_fixed_valid_score_distribution.png
D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts\fs03_fixed_threshold_valid.png
D:\ML_Trade_Bot\research\rf_mlflow\outputs\reproduce_fs03_fixed_params_1789400402\charts\fs03_fixed_threshold_test.png


## 19. Helper m? ph?ng l?nh theo threshold

In [32]:
def simulate_one_threshold_year(frame: pd.DataFrame, scores: np.ndarray, threshold: float, year: int) -> dict:
    year_arr = frame["year"].to_numpy()
    mask = year_arr == year

    chunk = frame.loc[mask].copy().reset_index(drop=True)
    score_chunk = scores[mask]

    orders = orders_from_scores(
        chunk,
        score_chunk,
        threshold,
        require_signal=True,
        session_filter="london_ny",
    )
    summary, results = simulate_orders(chunk, orders, tinh_tien=True)

    deals_path = CFG.outputs_dir / f"fs03_fixed_params_deals_t{threshold:.2f}_{year}.csv"
    pd.DataFrame(results).to_csv(deals_path, index=False, encoding="utf-8-sig")

    return {"threshold": threshold, "year": year, **summary, "deals_path": str(deals_path)}

## 20. Ch?y m? ph?ng

In [33]:
if SKIP_SIMULATE:
    yearly = pd.DataFrame()
    sim_total = pd.DataFrame()
    print("Skip simulation: SKIP_SIMULATE=True")
else:
    rows = []
    for threshold in SIMULATE_THRESHOLDS:
        for year in CFG.split.test_years:
            rows.append(simulate_one_threshold_year(frame, scores, threshold, year))

    yearly = pd.DataFrame(rows)
    sim_total = yearly.groupby("threshold", as_index=False).agg(
        deals=("deals", "sum"),
        resolved=("resolved", "sum"),
        wins=("wins", "sum"),
        losses=("losses", "sum"),
        no_res=("no_res", "sum"),
    )
    sim_total["winrate"] = (sim_total["wins"] * 100 / sim_total["resolved"].replace(0, np.nan)).round(4)

    yearly.to_csv(paths["yearly"], index=False, encoding="utf-8-sig")
    sim_total.to_csv(paths["sim_total"], index=False, encoding="utf-8-sig")

    display(yearly)
    display(sim_total)

D:\ML_Trade_Bot\vendor\scj\method\MoPhongDeals.py:71: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  self.RES_TINH_TIEN['start'][deal['start_index'] - 1] = deal['risk']
D:\ML_Trade_Bot\vendor\scj\method\MoPhongDeals.py:453: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves 

TypeError: Invalid value '5.84' for dtype 'int64'

## 21. V? chart m? ph?ng theo n?m

In [ ]:
if not SKIP_SIMULATE and not yearly.empty:
    for threshold in SIMULATE_THRESHOLDS:
        path = plot_yearly_mophong(
            yearly[yearly["threshold"] == threshold],
            chart_dir / f"fs03_fixed_mophong_t{threshold:.2f}.png",
        )
        chart_paths.append(path)
        print(path)

## 22. T?o summary JSON

In [ ]:
summary = {
    "run_id": run_id,
    "source_run": SOURCE_RUN_NAME,
    "feature_set": FEATURE_SET,
    "params": FIXED_PARAMS,
    "random_state": RANDOM_STATE,
    "skip_simulate": SKIP_SIMULATE,
    "valid_metrics": valid_metrics,
    "test_metrics": test_metrics,
    "simulated_thresholds": SIMULATE_THRESHOLDS,
    "simulation_total": sim_total.to_dict(orient="records") if not sim_total.empty else [],
    "curve_comparison_key_thresholds": key_threshold_comparison.to_dict(orient="records"),
    "out_dir": str(out_dir),
    "elapsed_sec": round(time.time() - started, 2),
}

with open(paths["summary"], "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("saved:", paths["summary"])
display(pd.Series(summary))

## 23. Log MLflow n?u kh? d?ng

In [ ]:
try:
    import mlflow

    mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}")
    mlflow.set_experiment(CFG.experiment_name)
    mlflow_available = True
except Exception as exc:
    mlflow_available = False
    print("MLflow unavailable:", exc)

if mlflow_available:
    with mlflow.start_run(run_name=f"reproduce_fs03_fixed_params_{run_id}"):
        mlflow.log_param("source_run", SOURCE_RUN_NAME)
        mlflow.log_param("feature_set", FEATURE_SET)
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_params({f"fixed_{k}": v for k, v in FIXED_PARAMS.items()})

        for k, v in valid_metrics.items():
            mlflow.log_metric(f"valid_{k}", v)
        for k, v in test_metrics.items():
            mlflow.log_metric(f"test_{k}", v)

        if not sim_total.empty:
            for _, row in sim_total.iterrows():
                t = str(row["threshold"]).replace(".", "_")
                mlflow.log_metric(f"mophong_t{t}_winrate", float(row["winrate"]))
                mlflow.log_metric(f"mophong_t{t}_deals", float(row["deals"]))

        for key in ["valid_curve", "test_curve", "comparison", "summary", "model"]:
            mlflow.log_artifact(str(paths[key]))

        if not SKIP_SIMULATE:
            for key in ["yearly", "sim_total"]:
                mlflow.log_artifact(str(paths[key]))

        for path in chart_paths:
            mlflow.log_artifact(str(path), artifact_path="charts")

    print("Logged to MLflow")

## 24. K?t qu? cu?i

In [ ]:
print(json.dumps(summary, ensure_ascii=False, indent=2))